# 8.4 • Which earlier observations does attention combine?

**Objectives:** calculate scaled dot-product attention; explain queries, keys, values and positional information; train one small Transformer-style block; compare against persistence, regression and an LSTM using the same forecast protocol.

**Prerequisites:** lessons 8.1 and 8.3; basic matrix multiplication. **Time:** 90–120 minutes. **Setup:** the same CPU PyTorch environment as lesson 8.3 (`requirements-lstm.txt` covers both). No new packages, GPU, credentials or pretrained model.

From the repo root, install base requirements and CPU PyTorch if needed:
```bash
python -m pip install -r requirements.txt
python -m pip install "torch>=2.2,<3" --index-url https://download.pytorch.org/whl/cpu
```
For other platforms use the [PyTorch installation selector](https://pytorch.org/get-started/locally/). Restart the kernel after installation.

**Data:** the same invented seasonal signal, disturbances and noise as lesson 8.3. This notebook is standalone; it does not require running another notebook or loading saved predictions. We make no environmental performance claims.

In [ ]:
from pathlib import Path
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from IPython.display import display
try:
    import torch
    from torch import nn
except ImportError as exc:
    raise ImportError('Install CPU PyTorch as described above, then restart the kernel.') from exc

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'course.py').exists())
OUT = ROOT / 'data' / 'generated' / 'attention'
OUT.mkdir(parents=True, exist_ok=True)
torch.set_num_threads(1)
torch.manual_seed(7)
print('PyTorch:', torch.__version__, '| CPU')

## Work through attention with three observations

A **query** is compared with each **key**. Their dot products, divided by the square root of the key dimension, become scores. Row-wise softmax converts scores into nonnegative weights summing to one. Each output is the weighted sum of **values**:

`attention(Q, K, V) = softmax(Q @ K.T / sqrt(d)) @ V`.

In self-attention, Q, K and V are learned projections of the same input sequence. For the worked calculation below, we choose them manually. Each row asks a question and each column supplies a candidate key/value. Predict which key the first query favors, then verify the weighted sum.

Attention weights describe a computation, not causal influence, physical importance or a complete explanation of the forecast. Values, projections, residual paths and the output layer also affect the prediction. See [Attention Is All You Need](https://arxiv.org/abs/1706.03762).

In [ ]:
Q = np.array([[1., 0.], [0., 1.], [1., 1.]])
K = np.array([[1., 0.], [0., 1.], [1., 1.]])
V = np.array([[10.], [20.], [30.]])
scores = Q @ K.T / np.sqrt(K.shape[1])
weights = np.exp(scores - scores.max(axis=1, keepdims=True))
weights /= weights.sum(axis=1, keepdims=True)
combined = weights @ V
display(pd.DataFrame(weights, index=['query 1', 'query 2', 'query 3'], columns=['key 1', 'key 2', 'key 3']))
print('First weighted sum:', weights[0], '@', V.ravel(), '=', combined[0, 0])
assert np.allclose(weights.sum(axis=1), 1) and (weights >= 0).all()
assert np.isclose(combined[0, 0], sum(weights[0, j] * V[j, 0] for j in range(3)))
fig, ax = plt.subplots(figsize=(4, 3))
im = ax.imshow(weights, cmap='Blues', vmin=0, vmax=1)
for i in range(3):
    for j in range(3): ax.text(j, i, f'{weights[i, j]:.2f}', ha='center', va='center')
ax.set(xticks=range(3), yticks=range(3), xlabel='Key/value index', ylabel='Query index', title='Hand-calculated attention')
fig.colorbar(im, ax=ax, label='Weight'); fig.tight_layout(); plt.show()

## Reuse the LSTM forecasting protocol

Split raw days into train 0–599, validation 600–799 and test 800–999 **before** making 24-day windows. Fit scaling on training observations only. All models predict the same next-step targets from the same observed history. First targets are days 24, 624 and 824; no raw observation occurs in multiple partitions.

This is rolling one-step forecasting with a fixed trained model. Earlier observed test values become available for later test windows. It is not a multi-step forecast issued at the beginning of the test period. Within-partition windows overlap; their errors are not independent.

All attention inputs are observations prior to the target. We allow bidirectional attention **within that historical window**, so a causal attention mask is unnecessary for this single next-step output. A model producing a prediction at every input timestep would require a different masking/data design.

In [ ]:
rng = np.random.default_rng(42)
time = np.arange(1000)
signal = 0.7 * np.sin(2 * np.pi * time / 60) + 0.2 * np.sin(2 * np.pi * time / 170)
disturbance = np.zeros_like(signal)
for start, amplitude in [(210, -0.8), (640, 0.6), (850, -0.7)]:
    age = time - start
    active = age >= 0
    disturbance[active] += amplitude * np.exp(-age[active] / 12)
signal = signal + disturbance + rng.normal(0, 0.08, len(time))
LOOKBACK = 24
SEGMENTS = {'train': (0, 600), 'validation': (600, 800), 'test': (800, 1000)}
scaler = StandardScaler().fit(signal[:600, None])
scaled = scaler.transform(signal[:, None]).ravel()

def make_windows(values, start, stop, lookback):
    if lookback < 1 or stop - start <= lookback:
        raise ValueError('Need a positive lookback shorter than the segment.')
    target_times = np.arange(start + lookback, stop)
    input_times = target_times[:, None] - np.arange(lookback, 0, -1)
    assert input_times.min() >= start and input_times.max() < stop
    assert np.all(input_times < target_times[:, None])
    return (values[input_times, None].astype(np.float32),
            values[target_times, None].astype(np.float32), target_times, input_times)

data = {name: make_windows(scaled, start, stop, LOOKBACK) for name, (start, stop) in SEGMENTS.items()}
used = {name: set(np.concatenate([entry[2], entry[3].ravel()])) for name, entry in data.items()}
assert used['train'].isdisjoint(used['validation'])
assert used['train'].isdisjoint(used['test']) and used['validation'].isdisjoint(used['test'])
assert np.isclose(scaler.mean_[0], signal[:600].mean())
display(pd.DataFrame([{'split': name, 'windows': len(entry[0]), 'first_target_day': entry[2][0],
                       'last_target_day': entry[2][-1]} for name, entry in data.items()]))
fig, ax = plt.subplots(figsize=(11, 3))
for name, (start, stop) in SEGMENTS.items():
    ax.plot(time[start:stop], signal[start:stop], label=name)
ax.set(xlabel='Synthetic day', ylabel='Arbitrary units', title='One synthetic signal, chronological partitions')
ax.legend(); fig.tight_layout(); plt.show()

## One small Transformer-style block

Project one scalar per timestep to an eight-dimensional representation. Add fixed sinusoidal position vectors so the network can distinguish early and late observations. Learn one set of Q/K/V projections, compute self-attention, then use a residual connection and layer normalization. A small feed-forward network, another residual connection and normalization complete the block. Mean pooling over timesteps feeds a one-value forecast head.

This is a **single-head encoder-style block**, not a language model or a full production Transformer. It has no tokenization, pretraining or decoder. We expose attention weights directly for teaching. Standard implementations offer multiple heads; see [PyTorch MultiheadAttention](https://docs.pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html).

Without positions, this model's self-attention is permutation-equivariant and mean pooling makes its forecast permutation-invariant. It can use the values present in the window but cannot distinguish their order. With positions, order can matter. Positional information helps express order; it does not guarantee better validation performance.

In [ ]:
class TinyAttention(nn.Module):
    def __init__(self, use_positions=True, width=8):
        super().__init__()
        self.use_positions, self.width = use_positions, width
        self.embed = nn.Linear(1, width)
        self.qkv = nn.Linear(width, 3 * width)
        self.mix = nn.Linear(width, width)
        self.norm1, self.norm2 = nn.LayerNorm(width), nn.LayerNorm(width)
        self.feedforward = nn.Sequential(nn.Linear(width, 16), nn.ReLU(), nn.Linear(16, width))
        self.head = nn.Linear(width, 1)
        position = torch.arange(LOOKBACK).float()[:, None]
        frequency = torch.exp(torch.arange(0, width, 2).float() * (-np.log(10000.0) / width))
        encoding = torch.zeros(LOOKBACK, width)
        encoding[:, 0::2] = torch.sin(position * frequency)
        encoding[:, 1::2] = torch.cos(position * frequency)
        self.register_buffer('position_encoding', encoding)

    def forward(self, sequence, return_attention=False):
        z = self.embed(sequence)
        if self.use_positions: z = z + self.position_encoding[:sequence.shape[1]]
        q, k, v = self.qkv(z).chunk(3, dim=-1)
        attention = torch.softmax(q @ k.transpose(-2, -1) / np.sqrt(self.width), dim=-1)
        z = self.norm1(z + self.mix(attention @ v))
        z = self.norm2(z + self.feedforward(z))
        forecast = self.head(z.mean(dim=1))
        return (forecast, attention) if return_attention else forecast

class SmallLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.memory = nn.LSTM(1, 8, batch_first=True)
        self.output = nn.Linear(8, 1)
    def forward(self, sequence):
        outputs, _ = self.memory(sequence)
        return self.output(outputs[:, -1, :])

X_train, y_train = [torch.from_numpy(a) for a in data['train'][:2]]
X_val, y_val = [torch.from_numpy(a) for a in data['validation'][:2]]
probe = TinyAttention()
with torch.no_grad():
    forecast, attention = probe(X_val[:2], return_attention=True)
assert forecast.shape == (2, 1) and attention.shape == (2, LOOKBACK, LOOKBACK)
assert torch.allclose(attention.sum(dim=-1), torch.ones(2, LOOKBACK), atol=1e-6)
assert torch.all(attention >= 0)
print('Trainable attention-model parameters:', sum(p.numel() for p in probe.parameters()))

## Train matched runs and inspect validation results

Use the same training windows, 160-epoch budget, Adam learning rate 0.01 and gradient clipping as lesson 8.3. Select each model's checkpoint using validation MSE only. We train positional attention, no-position attention and the small LSTM; this gives a direct baseline without depending on files from another lesson.

Matching epochs is a simple classroom comparison, not equal compute or an exhaustive hyperparameter search. Fix these settings before viewing the test results. Different initializations and time blocks may change rankings.

In [ ]:
def fit_model(factory, epochs=160, seed=7):
    torch.manual_seed(seed)
    model = factory()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.MSELoss()
    history, best_loss, best_state, best_epoch = [], np.inf, None, None
    for epoch in range(epochs + 1):
        model.eval()
        with torch.no_grad():
            training_loss = criterion(model(X_train), y_train).item()
            validation_loss = criterion(model(X_val), y_val).item()
        assert np.isfinite([training_loss, validation_loss]).all()
        history.append({'epoch': epoch, 'training_mse': training_loss, 'validation_mse': validation_loss})
        if validation_loss < best_loss:
            best_loss, best_epoch = validation_loss, epoch
            best_state = copy.deepcopy(model.state_dict())
        if epoch == epochs: break
        model.train(); optimizer.zero_grad()
        error = criterion(model(X_train), y_train)
        error.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
    model.load_state_dict(best_state); model.eval()
    return model, pd.DataFrame(history), best_epoch

factories = {'Attention + positions': lambda: TinyAttention(True),
             'Attention without positions': lambda: TinyAttention(False), 'LSTM': SmallLSTM}
models, histories, selection = {}, {}, []
for name, factory in factories.items():
    fitted, history, epoch = fit_model(factory)
    models[name], histories[name] = fitted, history
    assert history.training_mse.min() < history.training_mse.iloc[0]
    selection.append({'model': name, 'epoch': epoch, 'validation_mse': history.validation_mse.min(),
                      'parameters': sum(p.numel() for p in fitted.parameters())})
display(pd.DataFrame(selection))
fig, axes = plt.subplots(1, 3, figsize=(12, 3), constrained_layout=True)
for ax, (name, history) in zip(axes, histories.items()):
    ax.plot(history.epoch, history.training_mse, label='train')
    ax.plot(history.epoch, history.validation_mse, label='validation')
    ax.set(title=name, xlabel='Epoch', ylabel='Standardized MSE'); ax.legend()
fig.savefig(OUT / 'learning_curves.png', dpi=140); plt.show()
pd.DataFrame(selection).to_csv(OUT / 'validation_comparison.csv', index=False)

## Remove position information: a concrete invariance check

Reverse a validation window. Its values are unchanged but their order is different. The no-position model must return the same forecast up to floating-point rounding. The positional model can change its answer. This comparison is diagnostic; the reversed sequence no longer represents the actual observed history.

The heatmap below is from the positional model on the original validation window. Row t is the query at position t; columns show how its attention distributes across key/value positions. These are intermediate mixing weights, not a causal attribution of the final mean-pooled forecast.

In [ ]:
original = X_val[:1]
reversed_window = original.flip(dims=[1])
with torch.no_grad():
    no_position = models['Attention without positions']
    a, b = no_position(original), no_position(reversed_window)
    assert torch.allclose(a, b, atol=1e-5), 'No-position mean-pooled model should ignore order.'
    positional = models['Attention + positions']
    prediction, weights = positional(original, return_attention=True)
    reversed_prediction = positional(reversed_window)
print('No-position forecast difference:', float(torch.abs(a-b).item()))
print('Positional forecast difference:', float(torch.abs(prediction-reversed_prediction).item()))
assert torch.allclose(weights.sum(dim=-1), torch.ones(1, LOOKBACK), atol=1e-6)
lags = np.arange(-LOOKBACK, 0)
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
axes[0].plot(lags, original[0, :, 0].numpy(), marker='.')
axes[0].set(xlabel='Lag relative to target', ylabel='Standardized value', title='Observed validation history')
im = axes[1].imshow(weights[0].numpy(), cmap='viridis', origin='lower')
axes[1].set(xticks=[0, 11, 23], xticklabels=[-24, -13, -1],
            yticks=[0, 11, 23], yticklabels=[-24, -13, -1],
            xlabel='Key/value lag', ylabel='Query lag', title='Learned attention: mixing, not causation')
fig.colorbar(im, ax=axes[1], label='Attention weight')
fig.savefig(OUT / 'attention_heatmap.png', dpi=140); plt.show()

## Freeze choices, then compare on the same test days

Report MAE/RMSE in original arbitrary units for persistence, lagged ridge, LSTM and the positional attention model. The no-position model remains a validation ablation rather than another test-driven selection opportunity. All forecast targets and available observations are identical. The attention model need not outperform the simpler alternatives.

In [ ]:
ridge = Ridge(alpha=1.0).fit(data['train'][0][:, :, 0], data['train'][1].ravel())
test_windows, _, target_days, _ = data['test']
predictions = {'Persistence': test_windows[:, -1, 0],
               'Lagged ridge': ridge.predict(test_windows[:, :, 0])}
with torch.no_grad():
    for name in ['LSTM', 'Attention + positions']:
        predictions[name] = models[name](torch.from_numpy(test_windows)).numpy().ravel()
actual = signal[target_days]
rows, result = [], pd.DataFrame({'day': target_days, 'observed': actual})
for name, scaled_prediction in predictions.items():
    prediction = scaler.inverse_transform(scaled_prediction[:, None]).ravel()
    assert prediction.shape == actual.shape and np.isfinite(prediction).all()
    result[name] = prediction
    rows.append({'model': name, 'MAE': mean_absolute_error(actual, prediction),
                 'RMSE': np.sqrt(mean_squared_error(actual, prediction))})
assert np.allclose(result['Persistence'], signal[target_days-1], atol=1e-6)
metrics = pd.DataFrame(rows); display(metrics)
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(target_days, actual, color='black', linewidth=2, label='observed')
for name in predictions: ax.plot(target_days, result[name], label=name, alpha=0.8)
ax.set(xlabel='Synthetic day', ylabel='Arbitrary units', title='Rolling one-step forecasts using observed history')
ax.legend(ncol=3); fig.tight_layout(); fig.savefig(OUT / 'forecast_comparison.png', dpi=140); plt.show()
metrics.to_csv(OUT / 'test_metrics.csv', index=False)
result.to_csv(OUT / 'test_predictions.csv', index=False)

## Your independent challenge

1. Change one key in the three-observation example. Predict how the first query's weights and weighted value change; calculate and explain the result.
2. Interpret the positional ablation: compare validation loss, parameter counts and the reversal check. Explain why order invariance is an architectural property, not evidence that the model failed to train.
3. Try one new seed with the fixed architecture, comparing validation only. Does a single run justify a universal ranking?
4. Propose a 12-versus-24 day history experiment. Update positional encodings and rebuild windows, then compare on the intersection of target days using the original training-only scaler.

**Submit:** one attention calculation, a validation comparison, an annotated heatmap and a 100-word explanation of what attention weights do and do not establish. Once you have seen the test scores, reserve new data for later final-performance claims.

In [ ]:
# Your calculation or validation-only experiment here.

<details><summary>Solution guidance — open after trying</summary>

For Q[0]=[1,0], only a key's first coordinate affects its score; increasing it raises that key's weight while reducing other weights because each row must sum to one. The final result also depends on its value. The no-position mean-pooled model treats permutations as the same set of observations; the positional model can represent order but is not guaranteed to learn the best forecasting rule. For shorter history, align target-day arrays before scoring so differences do not simply reflect different evaluation days. A new seed is a robustness check, not a reason to select the luckiest test score.

</details>

## Check yourself and assessment

What are queries, keys and values? Why normalize each row? Why is unrestricted attention inside this historical window allowed? When would causal masking be required? Why is a bright heatmap cell not a causal explanation?

| Criterion | Points | Evidence |
|---|---:|---|
| Attention mechanics | 3 | Correct score, softmax and weighted-value calculation |
| Order and temporal integrity | 3 | Positional/reversal explanation; past-only windows and scaling |
| Independent analysis | 2 | Controlled validation comparison and aligned targets |
| Communication | 2 | Baselines, heatmap limitations and forecast scope |

**Mastery target:** 8/10 with no future information in inputs and no causal interpretation of attention weights.

## Connection to RAG and real environmental data

Many language models use attention, but RAG's retrieval stage and a model's internal attention are different operations. This notebook teaches an internal information-mixing mechanism, not document retrieval, language generation or a foundation model.

For environmental forecasting, retain dates, units, missingness and observation availability; define the forecast horizon and split by time and site as appropriate. Compare against persistence and seasonal baselines on common targets. Attention costs grow quadratically with window length in this implementation. More history and larger models add cost without guaranteeing useful information.

## Educator note

Suggested 100 minutes: 15 for hand attention, 15 for chronological windows, 20 for the small block, 20 for positional ablation/heatmap, and 30 for the independent exercise and baseline interpretation. Keep the question focused on how observations are combined, rather than expecting a Transformer to win.